# Qwen3.5 + SGLang GPU Demo

この Notebook は、SGLang で Qwen3.5 を GPU 上で起動し、テキスト入力と画像入力の両方を試すデモです。


## 1. 前提
- Docker が使える
- NVIDIA GPU が見えている
- Hugging Face のトークンが必要な場合は `HF_TOKEN` を設定する


In [ ]:
!nvidia-smi


In [ ]:
!python scripts/generate_test_image.py


## 2. SGLang サーバ起動
下のセルは Docker コンテナをバックグラウンド起動します。必要に応じてモデル名を変更してください。


In [ ]:
import os, shlex, subprocess
MODEL = os.environ.get('MODEL_NAME', 'Qwen/Qwen3.5-4B')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
cmd = f'''docker run -d --rm --name qwen35-sglang-demo --gpus all --ipc=host --shm-size 16g -p 30000:30000 -v {Path.cwd() / 'assets'}:/workspace/assets -v {Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface -e HF_TOKEN={shlex.quote(HF_TOKEN)} lmsysorg/sglang:latest-cu130-runtime python3 -m sglang.launch_server --model-path {shlex.quote(MODEL)} --host 0.0.0.0 --port 30000 --tp-size 1 --mem-fraction-static 0.8 --context-length 32768 --reasoning-parser qwen3'''
print(cmd)
subprocess.run(cmd, shell=True, check=False)


In [ ]:
!docker logs qwen35-sglang-demo --tail 200


## 3. テキスト入力テスト


In [ ]:
!python scripts/text_request.py --model Qwen/Qwen3.5-4B


## 4. 画像入力テスト


In [ ]:
!python scripts/vision_request.py --model Qwen/Qwen3.5-4B --image assets/demo_image.png


## 5. 後始末


In [ ]:
!docker rm -f qwen35-sglang-demo || true
